# Теоретический справочник: Реляционные связи, объединение таблиц и подготовка к SQL

Данное руководство содержит теоретический базис и практические примеры кода, необходимые для выполнения Практической работы №3:
1. **Реляционная модель данных**: зачем разделять данные на таблицы фактов и измерений.
2. **Первичные и внешние ключи (PK / FK)**: логика связывания сущностей.
3. **Все 4 типа соединений (`INNER`, `LEFT`, `RIGHT`, `FULL OUTER`)**: наглядный пример на тестовых данных с разбором работы функции `pd.merge()`.
4. **Проблема декартова произведения**: почему дубликаты ключей приводят к ошибкам в расчетах.
5. **Работа с неструктурированными данными (JSON)**:
   - Распаковка словарей через `pd.json_normalize()`;
   - Распаковка массивов списков через `.explode()`.
6. **Работа с базой данных SQLite в Python**: экспорт через `.to_sql()` и контрольные запросы через `pd.read_sql_query()`.


## 0. Инициализация библиотек


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import json

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)


## 1. Реляционная модель: факты и измерения

Хранение всех данных в одном плоском CSV-файле приводит к дублированию информации и ошибкам при ее изменении. В реляционной архитектуре данные разделяются на два типа таблиц:

1. **Таблицы измерений (справочники, Dimensions):**
   * Описывают статичные свойства объектов реального мира (клиенты, товары, компьютеры, тарифы).
   * Содержат уникальный идентификатор строки — **первичный ключ (Primary Key)**.
   * Примеры: `users` (`user_id`, `birth_date`, `gender`), `pcs` (`pc_id`, `pc_zone`, `gpu`).

2. **Таблицы фактов (журналы событий, Facts):**
   * Фиксируют события и транзакции, привязанные ко времени.
   * Ссылаются на таблицы измерений через **внешние ключи (Foreign Keys)**.
   * Примеры: `sessions` (`session_id`, `user_id`, `pc_id`, `date`, `hours_played`), `bar_sales` (`order_id`, `session_id`, `item_id`, `qty`).


## 2. Первичные и внешние ключи

* **Первичный ключ (Primary Key, PK):** столбец (или комбинация столбцов), значения которого строго уникальны и не могут быть пустыми (`NaN`). По первичному ключу можно однозначно идентифицировать любую запись справочника.
* **Внешний ключ (Foreign Key, FK):** столбец в таблице фактов, значения которого ссылаются на первичный ключ соответствующего справочника.

Связь между справочником и таблицей событий чаще всего является связью **«один-ко-многим» (1:N)**: один клиент может совершить много сессий, но у конкретной сессии клиент ровно один.


## 3. Четыре типа объединения (JOIN) в Pandas

В Pandas слияние двух таблиц по общему ключу выполняется с помощью метода `pd.merge(left_df, right_df, on='key', how='...')`.
Параметр `how` определяет, какие строки попадут в итоговый датафрейм.

Создадим минимальный тестовый пример из двух таблиц:
* `sessions` (3 сессии: две принадлежат клиенту USR-1, одна сыграна анонимно без профиля);
* `users` (2 клиента: USR-1 играл в клубе, USR-2 зарегистрирован, но сессий не имел).


In [2]:
# Тестовые данные для демонстрации
sessions_demo = pd.DataFrame({
    'session_id': [101, 102, 103],
    'user_id': ['USR-1', 'USR-1', np.nan],
    'hours_played': [3, 2, 1]
})

users_demo = pd.DataFrame({
    'user_id': ['USR-1', 'USR-2'],
    'name': ['Алексей', 'Борис'],
    'loyalty_tier': ['Gold', 'Silver']
})

print("Таблица сессий:")
display(sessions_demo)

print("\nСправочник пользователей:")
display(users_demo)


Таблица сессий:


,session_id,user_id,hours_played
0,101,USR-1,3
1,102,USR-1,2
2,103,NaN,1



Справочник пользователей:


,user_id,name,loyalty_tier
0,USR-1,Алексей,Gold
1,USR-2,Борис,Silver


### 3.1. Внутреннее объединение: `how='inner'`
Возвращает только те строки, ключ которых есть **одновременно в обеих таблицах**.
Сессия анонимного гостя (нет в `users`) и профиль Бориса (нет сессий) отсекаются.


In [3]:
inner_demo = pd.merge(sessions_demo, users_demo, on='user_id', how='inner')
display(inner_demo)


,session_id,user_id,hours_played,name,loyalty_tier
0,101,USR-1,3,Алексей,Gold
1,102,USR-1,2,Алексей,Gold


### 3.2. Левое внешнее объединение: `how='left'`
Сохраняет **все строки левой таблицы** (`sessions`). Если для строки нет пары в правой таблице, подставляются `NaN`.
Анонимная сессия 103 сохраняется, что позволяет не терять транзакции.


In [4]:
left_demo = pd.merge(sessions_demo, users_demo, on='user_id', how='left')
display(left_demo)


,session_id,user_id,hours_played,name,loyalty_tier
0,101,USR-1,3,Алексей,Gold
1,102,USR-1,2,Алексей,Gold
2,103,NaN,1,NaN,NaN


### 3.3. Правое внешнее объединение: `how='right'`
Сохраняет **все строки правой таблицы** (`users`). Если у пользователя не было сессий, поля из левой таблицы заполняются `NaN`.
Это позволяет сразу увидеть неактивных клиентов или неиспользуемое оборудование.


In [5]:
right_demo = pd.merge(sessions_demo, users_demo, on='user_id', how='right')
display(right_demo)


,session_id,user_id,hours_played,name,loyalty_tier
0,101.0,USR-1,3.0,Алексей,Gold
1,102.0,USR-1,2.0,Алексей,Gold
2,NaN,USR-2,NaN,Борис,Silver


### 3.4. Полное внешнее объединение: `how='outer'`
Сохраняет **все строки из обеих таблиц**. Несовпадения с любой стороны подтягиваются с `NaN`.
С параметром `indicator=True` Pandas добавляет служебный столбец `_merge`, показывающий источник строки (`both`, `left_only`, `right_only`).


In [6]:
outer_demo = pd.merge(sessions_demo, users_demo, on='user_id', how='outer', indicator=True)
display(outer_demo)


,session_id,user_id,hours_played,name,loyalty_tier,_merge
0,101.0,USR-1,3.0,Алексей,Gold,both
1,102.0,USR-1,2.0,Алексей,Gold,both
2,NaN,USR-2,NaN,Борис,Silver,right_only
3,103.0,NaN,1.0,NaN,NaN,left_only


## 4. Опасность дубликатов ключей: декартово произведение

Если в правой таблице по ключу слияния есть неуникальные записи, метод `pd.merge()` сгенерирует все возможные комбинации пар строк ($N \times M$). В результате датафрейм раздуется, а аналитические суммы исказятся.

Перед объединением всегда проверяйте уникальность ключа в таблице измерений:
```python
print("Ключ уникален:", df_users['user_id'].is_unique)
```


## 5. Работа с неструктурированными данными: JSON

Часто системы выгружают данные в виде текстовых JSON-строк внутри ячеек CSV.

### Сценарий 1: JSON-словарь в ячейке
Применяется функция `json.loads` для преобразования строки в словарь и `pd.json_normalize()` для разворачивания словаря в столбцы:


In [13]:
# Пример строки с характеристиками ПК
raw_df = pd.DataFrame({
    'pc_id': ['PC-01', 'PC-02'],
    'specs_json': [
        '{"gpu": "RTX 4060", "ram_gb": 32, "monitor_hz": 240}',
        '{"gpu": "RTX 3050", "ram_gb": 16, "monitor_hz": 165}'
    ]
})

# Распаковка словарей в столбцы
specs_list = [json.loads(s) for s in raw_df['specs_json']]
df_specs = pd.json_normalize(specs_list)
display(df_specs)

# Объединение с исходным идентификатором
df_pcs_clean = pd.concat([raw_df[['pc_id']], df_specs], axis=1)
display(df_pcs_clean)


,gpu,ram_gb,monitor_hz
0,RTX 4060,32,240
1,RTX 3050,16,165


,pc_id,gpu,ram_gb,monitor_hz
0,PC-01,RTX 4060,32,240
1,PC-02,RTX 3050,16,165


### Сценарий 2: JSON-массив (список объектов, отношение 1:N)
Когда в ячейке хранится список покупок (например, состав чека), это нарушает Первую нормальную форму (1NF).
Для разворачивания массива в отдельные строки используется связка `json.loads` + метод `.explode()`:


In [14]:
# Пример сессий со списком покупок
orders_raw = pd.DataFrame({
    'session_id': [201, 202],
    'user_id': ['USR-1', 'USR-2'],
    'bar_orders': [
        '[{"item": "Monster", "qty": 2}, {"item": "Чипсы", "qty": 1}]',
        '[]'
    ]
})

# 1. Парсинг строки в Python-список
orders_raw['bar_orders'] = orders_raw['bar_orders'].apply(lambda s: json.loads(s) if pd.notna(s) else [])
display(orders_raw)

# 2. Разворачивание списка на отдельные строки
orders_exploded = orders_raw.explode('bar_orders')
display(orders_exploded)


,session_id,user_id,bar_orders
0,201,USR-1,"[{'item': 'Monster', 'qty': 2}, {'item': 'Чипс..."
1,202,USR-2,[]


,session_id,user_id,bar_orders
0,201,USR-1,"{'item': 'Monster', 'qty': 2}"
0,201,USR-1,"{'item': 'Чипсы', 'qty': 1}"
1,202,USR-2,NaN


## 6. Выгрузка в SQLite и проверка контрольным SQL-запросом

Для сохранения очищенных данных в реляционную СУБД SQLite используется встроенная библиотека `sqlite3`:


In [9]:
# 1. Подключение к локальной базе данных
conn = sqlite3.connect(':memory:') # В учебных заданиях используйте имя файла, например 'club.db'

# 2. Экспорт датафреймов в таблицы базы данных
inner_demo.to_sql('user_sessions', conn, if_exists='replace', index=False)

# 3. Выполнение проверочного SQL-запроса через pd.read_sql_query
sql_query = """
SELECT loyalty_tier, COUNT(session_id) AS total_sessions, AVG(hours_played) AS avg_hours
FROM user_sessions
GROUP BY loyalty_tier;
"""
df_sql_result = pd.read_sql_query(sql_query, conn)
display(df_sql_result)

conn.close()


,loyalty_tier,total_sessions,avg_hours
0,Gold,2,2.5
